In [3]:
# import libraries
import pandas as pd
import json
import re
import sys
from seqeval.metrics import classification_report as seqeval_classification_report

# import custom helper functions
sys.path.append("../../utils/")
from preprocessing import create_regex_pattern
# from classification import tokenize_word_level, text_to_bio
from custom_evaluation import extract_spans#, mention_level_evaluation, sentence_level_evaluation

# import the annotations
with open("../../01_data/annotations_reduced.json", "r") as f:
    data = json.load(f)

# import the dictionary
group_dictionary_df = pd.read_csv("../../01_data/groups_dictionary.csv")

In [4]:
# function to convert the annotations to bio tags on the word level
def tokenize_word_level(sentence):
    
    # get all words' start and end index
    word_spans = []
    char_idx = 0

    # split sentence via regex which ensures to also split at punctuation
    words = re.findall(r"\w+|'\w+|[^\w\s]", sentence)

    for word in words:
        start_idx = sentence.find(word, char_idx)
        end_idx = start_idx + len(word)
        word_spans.append((start_idx, end_idx))
        char_idx = end_idx
    
    return words, word_spans

def text_to_bio(task):

    # extract sentence and annotations
    sentence = task["sentence"]
    annotations = task["annotations"]

    words, word_spans = tokenize_word_level(sentence)

    # initialize all tags as being O
    bio_tags = ["O"] * len(words)
    
    # loop through the annotations
    for annotation in annotations:
        start_ann, end_ann = annotation["start"], annotation["end"]
        for idx, (start_idx, end_idx) in enumerate(word_spans):
            if start_idx == start_ann:
                bio_tags[idx] = "B-sg"
            elif start_idx > start_ann and end_idx <= end_ann:
                bio_tags[idx] = "I-sg"

    return bio_tags

In [5]:
# add bio tags to the dataset
for task in data:
    bio_tags = text_to_bio(task)
    task["bio_tags"] = bio_tags

# create the regex pattern
combined_regex = create_regex_pattern(group_dictionary_df)

In [6]:
def find_dictionary_matches(sentence, dictionary_regex):
    
    # first tokenize the sentence
    words, word_spans = tokenize_word_level(sentence)

    bio_tags = ["O"] * len(words)

    for match in re.finditer(dictionary_regex, sentence, re.IGNORECASE):
        start_match, end_match = match.span()

        for idx, (start_idx, end_idx) in enumerate(word_spans):
            if start_idx == start_match:
                bio_tags[idx] = "B-sg"
            elif start_idx > start_match and end_idx <= end_match:
                bio_tags[idx] = "I-sg"
    
    return bio_tags

In [7]:
# quick fix, define all functions here again
def mention_level_evaluation_quickfix(all_true_spans, all_predicted_spans):

    # empty list to store all mention-level metrics
    span_metrics = []
    
    # loop through all sentences
    for sentence_preds, sentence_true in zip(all_predicted_spans, all_true_spans):
        # get all unique word ids for each span as a set
        pred_sets = [set(p) for p in sentence_preds]
        true_sets = [set(gt) for gt in sentence_true]
        # empty set that stores all visited true word ids
        matched_true_idx = set()

        # loop through all predicted spans
        for p_set in pred_sets:
            # variables that store with which span was the largest overlap
            best_overlap = 0
            best_idx = None
            # loop through true spans
            for i, t_set in enumerate(true_sets):
                # check overlap and store if it is a new best
                overlap = len(p_set & t_set)
                if overlap > best_overlap:
                    best_overlap = overlap
                    best_idx = i
            
            # if there was a match, calculate metrics for this predicted span
            if best_overlap > 0:
                t_set = true_sets[best_idx]
                precision = best_overlap / len(p_set)
                recall = best_overlap / len(t_set)
                f1 = (2*precision*recall)/(precision+recall)
                # mark the true span as visited
                matched_true_idx.add(best_idx)
            
            # if no match, assign 0 to all metrics
            else:
                precision, recall, f1 = 0.0, 0.0, 0.0
            span_metrics.append({"precision": precision,
                                 "recall": recall,
                                 "f1": f1})

        # loop through all true spans 
        for i, t_set in enumerate(true_sets):
            # if not already visited, assign 0 for all metrics
            if i not in matched_true_idx:
                span_metrics.append({"precision": 0.0,
                                     "recall": 0.0,
                                     "f1": 0.0})
    
    # take cross-span averages and return
    avg_precision = sum(m["precision"] for m in span_metrics) / len(span_metrics)
    avg_recall = sum(m["recall"] for m in span_metrics) / len(span_metrics)
    avg_f1 = sum(m["f1"] for m in span_metrics) / len(span_metrics)

    return {
        "precision": avg_precision,
        "recall": avg_recall,
        "f1": avg_f1
    }

def sentence_level_evaluation_quickfix(all_true_tags, all_predicted_tags):

    tp = fp = fn = 0

    for gt_tags, pred_tags in zip(all_true_tags, all_predicted_tags):
        has_true = any(tag.startswith(("B", "I")) for tag in gt_tags)
        has_pred = any(tag.startswith(("B", "I")) for tag in pred_tags)

        if has_true and has_pred:
            tp += 1
        elif has_pred and not has_true:
            fp += 1
        elif has_true and not has_pred:
            fn += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.00
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.00
    f1 = (2*precision*recall) / (precision + recall) if (precision + recall) > 0 else 0.00

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [13]:
# evaluate on the word level

# store all bio tags in a list
gt_bio_flat = [tag for sent in data for tag in sent["bio_tags"]]
gt_bio_nested = [sent["bio_tags"] for sent in data]

pred_bio_nested = []

for task in data:
    sentence = task["sentence"]
    pred_tags = find_dictionary_matches(sentence, combined_regex)
    pred_bio_nested.append(pred_tags)

# flatten the list
pred_bio_flat = [tag for sent in pred_bio_nested for tag in sent]


classification_report_seqeval = seqeval_classification_report(gt_bio_nested, pred_bio_nested, output_dict=True)
seqeval_metrics = {
      "precision": classification_report_seqeval["sg"]["precision"],
      "recall" : classification_report_seqeval["sg"]["recall"],
      "f1": classification_report_seqeval["sg"]["f1-score"]}

# evaluate on the entity level with custom cross-span metric
all_true_spans = []
all_predicted_spans = []

for idx in range(len(gt_bio_nested)):

    # get the spans
    all_true_spans.append(extract_spans(gt_bio_nested[idx]))
    all_predicted_spans.append(extract_spans(pred_bio_nested[idx]))
 
# apply cross-span evaluation
cross_span_metrics = mention_level_evaluation_quickfix(all_true_spans=all_true_spans, all_predicted_spans=all_predicted_spans)
sentence_level_metrics = sentence_level_evaluation_quickfix(gt_bio_nested, pred_bio_nested)

test_metrics = {}
test_metrics["dictionary_baseline"] = {
    "seqeval": seqeval_metrics,
    "cross_span": cross_span_metrics,
    "sentence_level": sentence_level_metrics
    }

with open("evaluation_metrics_dictionary.json", "w") as f:
    json.dump(test_metrics, f, indent=4)